<a href="https://colab.research.google.com/github/busycaesar/GPT/blob/Master/main-script.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# -*- coding: utf-8 -*-
"""main.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/github/busycaesar/GPT/blob/Master/main.ipynb
"""

# Download the dataset to train on.
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print("Total characters:", len(text))

# All the unique characters that occur in the text
unique_characters = sorted(list(set(text)))
vocab_size = len(unique_characters)

print("".join(unique_characters))
print(vocab_size)

# Mapping for each unique character.
stoi = { ch:i for i,ch in enumerate(unique_characters) }
itos = { i:ch for i,ch in enumerate(unique_characters) }

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(encode("hii there"))
print(decode(encode("hii there")))

import torch

encoded_text = encode(text)

dataset = torch.tensor(encoded_text, dtype=torch.long)

print(dataset.shape, dataset.dtype)
print(dataset[:1000])

# Split the data into train and validation sets

train_dataset_proportion = 0.9

number_of_dataset = int(train_dataset_proportion*len(dataset))

train_dataset = dataset[:number_of_dataset]
validation_dataset = dataset[number_of_dataset:]

block_size = 8

train_dataset[:block_size+1]

x = train_dataset[:block_size]
y = train_dataset[1:block_size+1]

for tensor in range(block_size):
    context = x[:tensor+1]
    target = y[tensor]
    print(f"When input is {context} the expected target is {target}.")

# To maintain reproducability of random indices.
torch.manual_seed(1337)

def get_batch(split, block_size, batch_size):
    dataset = train_dataset if split == 'train' else validation_dataset
    # Returns "batch_size" (4) random starting indices from the dataset.
    # The upper bound is "len(dataset) - block_size" (1003854 - 8) so that there are enough tokens for the block size even if the largest possible index is picked.
    ix = torch.randint(len(dataset) - block_size, (batch_size,))

    # Get "block_size" tokens starting at each chosen index, for all indices.
    context = torch.stack([dataset[i:i+block_size] for i in ix])

    # Get "block_size" tokens starting one position after each chosen index, for all indices.
    target = torch.stack([dataset[i+1:i+block_size+1] for i in ix])
    return context, target

# Parallel process on GPU
batch_size = 4
# Maximum context length for predicting next token
block_size = 8

context_ids, target_ids = get_batch('train', block_size, batch_size)
print('inputs:')
print(context_ids)
print()
print('targets:')
print(target_ids)

print('----')

for batch in range(batch_size):
    for tensor in range(block_size):
        context = context_ids[batch, :tensor+1]
        target = target_ids[batch, tensor]
        print(f"when input is {context.tolist()} the target: {target}")

import torch
import torch.nn as nn
from torch.nn import functional as F

# To maintain reproducability of random weights.
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # Lookup table mapping each token id to a row of vocab_size numbers. Each cell is assigned random values before training.
        # Normally, each token id is mapped to the row that contains the embedding (carrying semantic meaning) of the token.
        # The embeddings are then converted into logits to predict the next token.
        # Here, we skip all of that. The row length already equals vocab_size, so the row can be used directly as the logits.
        # So this model learns the logits directly in the table, instead of learning the many layers of weights that a real model uses to produce them.
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, context_ids, targets=None):
        # Gets the context batch matrix and returns logits array (vocab_size (64) size) for each index in the matrix.
        logits = self.token_embedding_table(context_ids) # Logit's Shape = Batch Size: B, Block Size: T, Vocab Size: C)

        if targets is None:
            return logits, None

        # Calculate the loss, based on the targets.

        B, T, C = logits.shape

        # Flattens the first two dimensions into one, so the shape goes from (Batch Size, Block Size, Vocab Size) to (Batch Size * Block Size, Vocab Size).
        #               Batch Item 1      Batch Item 2      Batch Item 3      Batch Item 4
        #               ________________  ________________  ________________  _________________
        # For example, [[[1, 2], [2, 3]], [[3, 4], [4, 5]], [[5, 6], [6, 7]], [[7, 8], [8, 9]]] (three nested arrays) becomes
        #               ______________  ______________  ______________  ______________
        #              [[1, 2], [2, 3], [3, 4], [4, 5], [5, 6], [6, 7], [7, 8], [8, 9]] (two nested arrays).
        # Essentially, the rows of all the batch items are joined into one flat list.
        # This is needed because the loss function expects the logits in the shape (Predictions, Vocab Size).
        logits = logits.view(B*T, C)

        # Converting the target matrix also into the same dimension as logits.
        targets = targets.view(B*T)

        loss = F.cross_entropy(logits, targets)

        print(loss)

        # The cross enthropy gets the first item from the targets array, which is the expected index of the next token id.
        next_token_id_index = targets[0]

        # Then it gets the first logit array, in which it checks for the value on the expected index of the next token id.
        value_at_expected_index = logits[0][next_token_id_index]

        print()
        print("Next expected token id's index   :",next_token_id_index)
        print("Logit value at the expected index:",value_at_expected_index)
        print()

        # The cross enthropy then applies softmax to all the items in the array at 0th index of the logits array.
        # Then gets the softmaxed value (probability) at the expected index, and applies negative log to the probability.
        # Log of any probability (0-1) produces a negative number.
        # For higher probability (e.g., 0.9), log returns smaller negative (e.g., -0.105) and vise-versa. Negation converts to positive: -log(prob) gives positive loss.
        # For higher probability, we get lower loss value and vise-versa. So the correct predictions are rewarded.
        # It calculates the loss for each token prediction and returns the average loss.

        return logits, loss

    def generate(self, context_ids, max_new_tokens):
        # context_ids' shape = (B, T). Array of B Batch Items. Each item has T context ids.
        # context_ids = [[5, 2, 8], <== Batch 1
        #                [7, 1, 9]] <== Batch 2
        #                 ^  ^  ^
        #                 |  |  |
        #                 T  T  T
        #                 O  O  O
        #                 K  K  K
        #                 E  E  E
        #                 N  N  N
        #
        #                 1  2  3
        #
        # Shape: (2, 3)
        for _ in range(max_new_tokens):
            logits, _ = self(context_ids)

            # Extract logits of the last token index for each batch.
            # Example: [[0.1, -0.5,  0.3, ..., 0.2],  <== Batch 1; Logits for token id index 8
            #           [0.2,  0.1, -0.1, ..., 0.4]]  <== Batch 2; Logits for token id index 9
            last_token_index_logits = logits[:, -1, :]

            # Get the probabilities by applying softmax to all the logits of the last token index.
            # Example: [[0.02, 0.01, 0.05, ..., 0.03], <== Batch 1; Probabilities for each token id index, after token id index 8
            #           [0.03, 0.02, 0.01, ..., 0.04]] <== Batch 2; Probabilities for each token id index, after token id index 9
            probabilities = F.softmax(last_token_index_logits, dim=-1)

            # Returns indices of the next predicted token, for all the batches.
            # Example: [[42], <== Batch 1: Next token id index, after index 8
            #           [15]] <== Batch 2: Next token id index, after index 9
            indices_of_next_token = torch.multinomial(
                probabilities,
                # Randomly returns 1 token id index per batch based on the probability distribution.
                num_samples=1
            )

            # Append the predicted index for each batch at the end.
            # Updated: [[5, 2, 8, 42],
            #           [7, 1, 9, 15]]
            context_ids = torch.cat((context_ids, indices_of_next_token), dim=1)
        return context_ids

m = BigramLanguageModel(vocab_size)
print("Context Id Batch")
print(context_ids)
print()
print("Target Id Batch")
print(target_ids)
print()
logits, loss = m(context_ids, target_ids)
print(logits.shape)
print(loss)
print()

def generate_next_tokens(context_ids, max_new_tokens=100):
    # Generate next tokens using the given context ids.
    generated_tokens = m.generate(context_ids, max_new_tokens)
    print(generated_tokens)

    # Get the tokens for the first batch.
    generated_tokens_batch_1 = generated_tokens[0]

    # Convert the tokens array from tensor to list.
    token_id_index_list = generated_tokens_batch_1.tolist()
    print(token_id_index_list)

    # Use the tokens list to decode it using tokenizer.
    generated_text = decode(token_id_index_list)

    return generated_text

initial_context = torch.zeros((1, 1), dtype=torch.long)
print(initial_context)

generated_next_tokens = generate_next_tokens(initial_context, 100)
print(generated_next_tokens)

# Create an AdamW optimizer object (a specific optimization algorithm).
optimizer = torch.optim.AdamW(
                # Pass all model parameters (token_embedding_table which contains weights) to be updated during training.
                m.parameters(),
                # Learning rate: controls how much weights change at every update step.
                # Lower lr means slower but more stable learning while, Higher lr means faster but may diverge.
                # new_weight = old_weight - (learning_rate * gradient)
                lr=1e-3
            )

batch_size = 32

for steps in range(1000):
    context_ids, target_ids = get_batch('train', block_size, batch_size)
    logits, loss = m(context_ids, target_ids)

    # Each cell in the token embedding table contains weights and each weights also has the property gradient.
    # The gradient is the derivative of loss with respect of weight. Hence, it indicates how much does the loss change by a minor change in weights.
    # The increase and decrease in the value of each weights is decided by this gradient.

    # We need to ensure that before we start calculating and storing the values in the gradient property of these weights, all the existing gradient properties in each weight of the token embedding table are cleared.
    # In short, it should not have the values calculate in the previous iteration.
    optimizer.zero_grad(set_to_none=True)

    # The loss goes back to each weight to calculate how much did that weight contributed into the loss.
    # Based on that calculates the gradient for that weights and stored the value in the gradient property of the weight.
    loss.backward()

    # This uses the calculated gradient value and update each weight cell in the token embedding table.
    optimizer.step()

# Prints the loss calculate during the last iteration.
print(loss.item())

generated_next_tokens = generate_next_tokens(initial_context, 1000)
print(generated_next_tokens)

Streaming output truncated to the last 5000 lines.

tensor(4.7101, grad_fn=<NllLossBackward0>)

Next expected token id's index   : tensor(43)
Logit value at the expected index: tensor(-0.4684, grad_fn=<SelectBackward0>)

tensor(4.7137, grad_fn=<NllLossBackward0>)

Next expected token id's index   : tensor(43)
Logit value at the expected index: tensor(0.6893, grad_fn=<SelectBackward0>)

tensor(4.6869, grad_fn=<NllLossBackward0>)

Next expected token id's index   : tensor(40)
Logit value at the expected index: tensor(-0.1272, grad_fn=<SelectBackward0>)

tensor(4.7001, grad_fn=<NllLossBackward0>)

Next expected token id's index   : tensor(52)
Logit value at the expected index: tensor(0.4993, grad_fn=<SelectBackward0>)

tensor(4.7183, grad_fn=<NllLossBackward0>)

Next expected token id's index   : tensor(58)
Logit value at the expected index: tensor(-0.6455, grad_fn=<SelectBackward0>)

tensor(4.7156, grad_fn=<NllLossBackward0>)

Next expected token id's index   : tensor(41)
Logit value at 